In [ ]:
import sys
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Optional, List, Dict, Tuple, Union

# Set SVG rendering
%config InlineBackend.figure_format = 'svg'

# Add the current directory to sys.path so we can import log_parser
sys.path.append(os.getcwd())

from log_parser import get_infos_from_dir, Info

In [ ]:
def get_experiment_df(target_dirs: Union[str, List[str]]) -> pd.DataFrame:
    """
    Scans the directories for logs and returns a processed DataFrame.
    """
    if isinstance(target_dirs, str):
        target_dirs = [target_dirs]
        
    all_infos = []
    for d in target_dirs:
        # Resolve path
        if os.path.isabs(d):
            full_path = d
        else:
            full_path = os.path.join(os.getcwd(), d)
            
        print(f"Scanning directory: {full_path}")
        if not os.path.exists(full_path):
             print(f"Warning: Directory {full_path} does not exist.")
             continue
             
        infos = get_infos_from_dir(full_path)
        print(f"Found {len(infos)} log files in {d}.")
        all_infos.extend(infos)
    
    print(f"Total log files found: {len(all_infos)}")
    
    data = [info.to_dict() for info in all_infos]
    if not data:
        return pd.DataFrame()
        
    df = pd.DataFrame(data)
    
    # Reorder columns if they exist
    cols = ["task_id", "method", "model", "bsz", "input_len", "output_len", 
            "page_size", "mem_frac", "ctx", "status", "throughput", 
            "topk", "est", "wind", "sink"]
    existing_cols = [c for c in cols if c in df.columns]
    remaining_cols = [c for c in df.columns if c not in existing_cols and c != "log_path"]
    
    df = df[existing_cols + remaining_cols]
    if "method" in df.columns and "bsz" in df.columns:
        df.sort_values(by=["method", "bsz"], inplace=True)
    
    return df

In [ ]:
# Mapping dictionary for formal labels
LABEL_MAPPING = {
    "bsz": "Batch Size",
    "input_len": "Input Length",
    "output_len": "Output Length",
    "page_size": "Page Size",
    "mem_frac": "Memory Fraction",
    "ctx": "Context Length",
    "throughput": "Throughput\n(token/s)",
    "model": "Model",
    "method": "Method",
    "topk": "Quest Top-K",
    "est": "Quest Estimate Kernel",
    "wind": "StreamingLLM Window Size",
    "sink": "StreamingLLM Sink Size"
}

In [ ]:
def plot_throughput(
    df: pd.DataFrame,
    x_col: str,
    hue_col: str,
    y_col: str = "throughput",
    title: str = None,
    xlabel: str = None,
    ylabel: str = None,
    show_range: bool = True,
    figsize: Tuple[int, int] = (10, 7),
    filepath: str = None,
    bar_width: float = 0.7
):
    """
    Generic plotting function for throughput bar charts with refined aesthetics.
    """
    # Set font scale globally for this plot context
    sns.set_context("notebook", font_scale=1.2)
    
    # Define custom color palette: Blue, Red, Yellow, Green
    custom_colors = ["#4C72B0", "#C44E52", "#E6AB02", "#55A868"]
    unique_hues = df[hue_col].unique()
    if len(unique_hues) > len(custom_colors):
        palette = sns.color_palette("tab10")
    else:
        palette = custom_colors[:len(unique_hues)]

    method2color = {
        "base": "#4C72B0",
        "quest": "#C44E52",
        "stream": "#E6AB02",
    }
    if hue_col == "method":
        palette = [method2color[m] for m in unique_hues]

    fig, ax = plt.subplots(figsize=figsize)
    
    # Configure error bars
    errorbar = 'sd' if show_range else None
    
    # Check if x-axis is numeric
    is_numeric = pd.api.types.is_numeric_dtype(df[x_col])

    if is_numeric:
        # Treat as numeric: use actual values for positioning
        sns.barplot(
            data=df, 
            x=x_col, 
            y=y_col, 
            hue=hue_col, 
            errorbar=errorbar, 
            capsize=0.1,
            palette=palette,
            ax=ax,
            edgecolor='white',
            linewidth=1,
            width=bar_width,
            native_scale=True # Use native scale for numeric x-axis
        )
        # Set x-ticks to exactly match the unique numeric values present in the data
        unique_x_vals = sorted(df[x_col].unique())
        ax.set_xticks(unique_x_vals)
        ax.set_xticklabels(unique_x_vals)
    else:
        # Categorical behavior (default)
        sns.barplot(
            data=df, 
            x=x_col, 
            y=y_col, 
            hue=hue_col, 
            errorbar=errorbar, 
            capsize=0.1,
            palette=palette,
            ax=ax,
            edgecolor='white',
            linewidth=1,
            width=bar_width
        )
    
    # 3. Grid: 50% transparent dashed lines
    ax.grid(True, axis='y', linestyle='--', alpha=0.5, color='gray')
    ax.set_axisbelow(True)

    # 4. Remove top and right spines
    sns.despine(top=True, right=True)
    
    # Axis color settings
    axis_color = '#CCCCCC' # Lighter gray
    
    # 1. Customizing axes colors 
    ax.spines['left'].set_color(axis_color) 
    ax.spines['left'].set_linewidth(2)
    ax.spines['left'].set_alpha(1.0)
    
    ax.spines['bottom'].set_color(axis_color)
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['bottom'].set_alpha(1.0)
    
    # Also color ticks to match, but labels are black
    ax.tick_params(axis='both', colors=axis_color, width=2, labelcolor='black')

    # Title setup
    if title:
        ax.set_title(title, pad=40, fontsize=16, fontweight='bold', color='#333333')

    # Labels
    x_lbl = xlabel if xlabel else LABEL_MAPPING.get(x_col, x_col)
    y_lbl = ylabel if ylabel else LABEL_MAPPING.get(y_col, y_col)
    
    ax.set_xlabel(x_lbl, fontsize=16, labelpad=10)
    
    # 6. Y-axis label on top, horizontal, shifted right
    ax.set_ylabel(y_lbl, rotation=0, loc='top', fontsize=16)
    ax.yaxis.set_label_coords(x=0.01, y=1.02) 
    
    # 2. Extend X-axis limit slightly on the right
    xmin, xmax = ax.get_xlim()
    ax.set_xlim(xmin, xmax + (xmax - xmin) * 0.05)
    
    # 3. Legend customization
    leg = ax.legend(
        title=None, 
        loc='upper center', 
        bbox_to_anchor=(0.5, 1.15), 
        ncol=len(unique_hues), 
        frameon=False,
        fontsize=16,
        handleheight=2, 
        handlelength=2, 
    )

    if filepath:
        plt.savefig(filepath, bbox_inches='tight')
        # also save as png with same prefix
        plt.savefig(filepath.replace(".pdf", ".png"), bbox_inches='tight')

    plt.tight_layout()
    plt.show()

In [ ]:
# Load data
target_dir = ["1225/batch", "1309/batch/stream"]
df = get_experiment_df(target_dir)

# Filter for successful runs
plot_df = df[df["status"] == "ok"].copy()

# Plot
plot_throughput(
    df=plot_df,
    x_col="bsz",
    hue_col="method",
    title="",
    show_range=False,
    figsize=(10, 6),
    filepath="assets/batch-mha-4b.pdf"
)

In [ ]:
# Load data
target_dir = "1205/batch-1b"
df = get_experiment_df(target_dir)

# Filter for successful runs
plot_df = df[df["status"] == "ok"].copy()

# Plot
plot_throughput(
    df=plot_df,
    x_col="bsz",
    hue_col="method",
    title="",
    show_range=False,
    figsize=(10, 6),
    filepath="assets/batch-mha-4b.pdf"
)

In [ ]:
# Load data
df = get_experiment_df(["1205/batch-1b", "1206/batch-1b-stream256"])

# Filter for successful runs
plot_df = df[df["status"] == "ok"].copy()
plot_df = plot_df[plot_df["wind"] != 512]

# Plot
plot_throughput(
    df=plot_df,
    x_col="bsz",
    hue_col="method",
    title="",
    show_range=False,
    figsize=(10, 6),
    filepath="assets/batch-mha-4b.pdf"
)

In [ ]:
# Load data
df = get_experiment_df("1206/input")

# Filter for successful runs
plot_df = df[df["status"] == "ok"].copy()

# Plot
plot_throughput(
    df=plot_df,
    x_col="input_len",
    hue_col="method",
    title="",
    show_range=False,
    figsize=(10, 6),
    filepath="assets/input.pdf"
)

In [ ]:
# Load data
df = get_experiment_df("1206/ctx")

# Filter for successful runs
plot_df = df[df["status"] == "ok"].copy()

# Plot
plot_throughput(
    df=plot_df,
    x_col="ctx",
    hue_col="method",
    title="",
    show_range=False,
    figsize=(10, 6),
    filepath="assets/ctx.pdf"
)

In [ ]:
# Load data
df = get_experiment_df("1206/quest/topk-more")

# Filter for successful runs
plot_df = df[df["status"] == "ok"].copy()

# Plot
plot_throughput(
    df=plot_df,
    x_col="topk",
    hue_col="method",
    title="",
    show_range=False,
    figsize=(10, 6),
    filepath="assets/quest-topk.pdf",
    bar_width=0.4
)

In [ ]:
# Load data
df = get_experiment_df("1206/stream/wind-more")

# Filter for successful runs
plot_df = df[df["status"] == "ok"].copy()

# Plot
plot_throughput(
    df=plot_df,
    x_col="wind",
    hue_col="method",
    title="",
    show_range=False,
    figsize=(10, 6),
    filepath="assets/stream-wind.pdf",
    bar_width=0.4
)